# Pandas DataFrame Basics

A compact, runnable reference covering the most-used DataFrame operations: creation, inspection, indexing/selection, additions/removals, sorting, grouping, joins, and I/O.

## Setup — imports and sample DataFrame

In [2]:
import pandas as pd
import numpy as np
np.random.seed(0)
df = pd.DataFrame({
  'id': range(1,6),
  'name': ['alice','bob','carol','dave','eva'],
  'age': [25, 32, 37, 29, 41],
  'score': np.random.randint(60,101, size=5)
})
df

,id,name,age,score
0,1,alice,25,60
1,2,bob,32,63
2,3,carol,37,63
3,4,dave,29,99
4,5,eva,41,69


## Inspecting data — shape, head/tail, info, dtypes

In [4]:
print('shape:', df.shape)
print('head:', df.head())
print('tail:', df.tail())
print('info:'); df.info()
print('dtypes:', df.dtypes)


shape: (5, 4)
head:    id   name  age  score
0   1  alice   25     60
1   2    bob   32     63
2   3  carol   37     63
3   4   dave   29     99
4   5    eva   41     69
tail:    id   name  age  score
0   1  alice   25     60
1   2    bob   32     63
2   3  carol   37     63
3   4   dave   29     99
4   5    eva   41     69
info:
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      5 non-null      int64
 1   name    5 non-null      str  
 2   age     5 non-null      int64
 3   score   5 non-null      int64
dtypes: int64(3), str(1)
memory usage: 292.0 bytes
dtypes: id       int64
name       str
age      int64
score    int64
dtype: object


## Accessing columns and rows — `[]`, dot, `loc`, `iloc`, boolean masks

In [5]:
# column access
print(df['name'])
print(df.name)  # shorthand (avoid if name shadows method)
# row access by label (loc) and position (iloc)
print('loc row 0:', df.loc[0])
print('iloc row 0:', df.iloc[0])
# boolean filtering
print('age > 30:', df[df['age'] > 30])
# select subset of columns and rows
print('subset:', df.loc[df['score']>80, ['id','name','score']])


0    alice
1      bob
2    carol
3     dave
4      eva
Name: name, dtype: str
0    alice
1      bob
2    carol
3     dave
4      eva
Name: name, dtype: str
loc row 0: id           1
name     alice
age         25
score       60
Name: 0, dtype: object
iloc row 0: id           1
name     alice
age         25
score       60
Name: 0, dtype: object
age > 30:    id   name  age  score
1   2    bob   32     63
2   3  carol   37     63
4   5    eva   41     69
subset:    id  name  score
3   4  dave     99


## Adding, modifying, and removing columns

In [6]:
# add new column vectorized
df['passed'] = df['score'] >= 70
print(df)
# compute derived column with assign (chaining)
df2 = df.assign(score_pct = lambda d: (d.score - d.score.min()) / (d.score.max() - d.score.min()))
print('with score_pct:', df2)
# drop a column (returns copy)
print('drop passed:', df2.drop(columns=['passed']))


   id   name  age  score  passed
0   1  alice   25     60   False
1   2    bob   32     63   False
2   3  carol   37     63   False
3   4   dave   29     99    True
4   5    eva   41     69   False
with score_pct:    id   name  age  score  passed  score_pct
0   1  alice   25     60   False   0.000000
1   2    bob   32     63   False   0.076923
2   3  carol   37     63   False   0.076923
3   4   dave   29     99    True   1.000000
4   5    eva   41     69   False   0.230769
drop passed:    id   name  age  score  score_pct
0   1  alice   25     60   0.000000
1   2    bob   32     63   0.076923
2   3  carol   37     63   0.076923
3   4   dave   29     99   1.000000
4   5    eva   41     69   0.230769


## Sorting and ranking

In [7]:
print(df.sort_values('score', ascending=False))
print('add rank column:', df.assign(rank=df['score'].rank(ascending=False)))


   id   name  age  score  passed
3   4   dave   29     99    True
4   5    eva   41     69   False
1   2    bob   32     63   False
2   3  carol   37     63   False
0   1  alice   25     60   False
add rank column:    id   name  age  score  passed  rank
0   1  alice   25     60   False   5.0
1   2    bob   32     63   False   3.5
2   3  carol   37     63   False   3.5
3   4   dave   29     99    True   1.0
4   5    eva   41     69   False   2.0


## Grouping & aggregation — `groupby` + `agg`/`transform`

In [8]:
# small example with city and values
dfg = pd.DataFrame({'city':['ny','la','ny','la','ny'], 'val':[10,20,30,40,50]})
print(dfg.groupby('city').agg(total=('val','sum'), mean=('val','mean')))
# transform preserves index (useful for computing per-row group features)
print('with group-mean col:', dfg.assign(group_mean=dfg.groupby('city')['val'].transform('mean')))


      total  mean
city             
la       60  30.0
ny       90  30.0
with group-mean col:   city  val  group_mean
0   ny   10        30.0
1   la   20        30.0
2   ny   30        30.0
3   la   40        30.0
4   ny   50        30.0


## Joins and concatenation — `merge`, `join`, `concat`

In [9]:
left = pd.DataFrame({'id':[1,2,3], 'A':[10,20,30]})
right = pd.DataFrame({'id':[2,3,4], 'B':[200,300,400]})
print('merge inner:', pd.merge(left, right, on='id', how='inner'))
print('merge left:', pd.merge(left, right, on='id', how='left'))
print('concat rows:', pd.concat([left, right], ignore_index=True))


merge inner:    id   A    B
0   2  20  200
1   3  30  300
merge left:    id   A      B
0   1  10    NaN
1   2  20  200.0
2   3  30  300.0
concat rows:    id     A      B
0   1  10.0    NaN
1   2  20.0    NaN
2   3  30.0    NaN
3   2   NaN  200.0
4   3   NaN  300.0
5   4   NaN  400.0


## Missing data — isna, dropna, fillna, interpolate

In [10]:
dft = pd.DataFrame({'a':[1,None,3], 'b':[None,2,3]})
print(dft.isna())
print('dropna:', dft.dropna())
print('fillna:', dft.fillna(0))
print('interpolate:', dft.interpolate())


       a      b
0  False   True
1   True  False
2  False  False
dropna:      a    b
2  3.0  3.0
fillna:      a    b
0  1.0  0.0
1  0.0  2.0
2  3.0  3.0
interpolate:      a    b
0  1.0  NaN
1  2.0  2.0
2  3.0  3.0


In [14]:
dft = pd.DataFrame({'a':[1,None,3], 'b':[None,2,3]})
print(dft.isna())
dft.dropna()
dft
dft.interpolate()
dft

       a      b
0  False   True
1   True  False
2  False  False


,a,b
0,1.0,NaN
1,NaN,2.0
2,3.0,3.0


## Common utilities: describe, value_counts, sample, to_csv

In [ ]:
print(df.describe())
print('
value_counts:
', df['age'].value_counts())
print('
sample:
', df.sample(2))
# write to CSV (example): df.to_csv('out.csv', index=False)


## Copy vs view gotcha and inplace note

In [ ]:
a = df[['id','age']]
b = a  # both reference same object until a copy is made
b.loc[0,'age'] = 99
print('df changed via view?
', df.head())
# to avoid: use copy()
c = df[['id','age']].copy()
c.loc[0,'age'] = 10
print('
original unchanged when using copy:
', df.head())


---
**Next steps:** I created `pandas_dataframe_basics_practise.ipynb` with core examples. Would you like me to run the cells to verify outputs, add more real-world examples, or include visualizations? 